In [1]:
%load_ext autoreload
%autoreload 2

from zrp import ZRP
import pandas as pd
import numpy as np

mapping = {
    "FIRST_NAME": "first_name", 
    "MIDDLE_NAME": "middle_name", 
    "LAST_NAME": "last_name", 
    "HOUSE_NUMBER": "house_number", 
    "STREET_ADDRESS": "street_address", 
    "CITY": "city", 
    "STATE": "state", 
    "ZIP_CODE": "zip_code",
    "RECID": "ZEST_KEY",
}

df = pd.read_parquet("data/temp.parquet")

df["ADDRESS"] = (
    df["ADDRESS"]
    .astype(str)
    .str.replace(r"[^\x00-\x7F]+", "", regex=True)
)

df["address_split"] = df["ADDRESS"].str.split(" ")

df["HOUSE_NUMBER"] = (
    df["address_split"]
    .str[0]
    .str.strip()
)

# keep only numeric house numbers
df["HOUSE_NUMBER"] = (
    pd.to_numeric(df["HOUSE_NUMBER"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["STREET_ADDRESS"] = np.where(
    df["HOUSE_NUMBER"].isna(),
    df["ADDRESS"],
    df["address_split"].str[1:].str.join(" ")
)

df["ZIP_CODE"] = df["ZIP_CODE"].astype("string")
df["RECID"] = df["RECID"].astype("string")

for c in ("CITY", "STATE", "ZIP_CODE", "HOUSE_NUMBER", "STREET_ADDRESS"):
    df[c] = df[c].replace("", pd.NA)
    
df = df.rename(columns=mapping)

df = df[list(mapping.values())]

mask = ~(
    df["house_number"].isna()
    & df["city"].isna()
    & df["state"].isna()
    & (
        df["street_address"].isna()
        | (df["street_address"] == "CONFIDENTIAL")
    )
)

df = df.loc[mask].reset_index(drop=True)

df.columns


Index(['first_name', 'middle_name', 'last_name', 'house_number',
       'street_address', 'city', 'state', 'zip_code', 'ZEST_KEY'],
      dtype='object')

In [5]:
from pathlib import Path
folder = Path('./artifacts')
for file in folder.iterdir():
	file.unlink()

zest_race_predictor = ZRP()
zest_race_predictor.fit()
zrp_output = zest_race_predictor.transform(df)
zrp_output

Directory already exists
####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 10000
     Is key unique: True
Directory already exists
   [Completed] Validating input data

   Formatting P1


  0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1230 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 1780 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 2430 tasks      | elapsed:    0.1s


   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['NC']
   ... on state: NC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Done 3180 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 4980 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 6030 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 7180 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 8430 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 9780 tasks      | elapsed:    0.3s
100%|██████████| 10000/10000 [00:00<00:00, 32365.93it/s]
[Parallel(n_jobs=-1)]: Done 10000 out of 10000 | elapsed:    0.3s finished


      ...replicating address
         ...Base
         ...Number processing...
         House number dataframe expansion is complete! (n=10014)
         ...Base
         ...Map street suffixes...
         ...Mapped & split by street suffixes...
         ...Number processing...

         Address dataframe expansion is complete! (n=13163)
      ...formatting
   [Completed] Processing geo data
   [Start] Mapping geo data
      ...merge user input & lookup table
      ...mapping


...

   [Completed] Validating input geo data
Directory already exists
...Output saved
   [Completed] Mapping geo data

[Completed] Preparing geo data

[Start] Preparing ACS data
   [Start] Validating ACS input data
     Number of observations: 10000
     Is key unique: True

   [Completed] Validating ACS input data

   ...loading ACS lookup tables


   ... combining ACS & user input data
 ...Copy dataframes
 ...Block group
 ...Census tract
 ...Zip code
 ...No match
 ...Merge
 ...Merging complete
[Complete] Preparing ACS data

   [Start] Validating pipeline input data
     Number of observations: 34021
     Is key unique: False
   [Completed] Validating pipeline input data



100%|██████████| 1/1 [00:00<00:00, 2053.01it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 3771.86it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished
100%|██████████| 1/1 [00:00<00:00, 4266.84it/s]
[Parallel(n_jobs=-1)]: Done   1 out of   1 | elapsed:    0.0s finished


   ...Proxies generated
...Output saved
...Output saved


,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,ZEST_KEY,AAPI,AIAN,BLACK,HISPANIC,WHITE,race_proxy,source_zrp_block_group,source_zrp_census_tract,source_zrp_zip_code
0,CHRISTINA,CASTAGNA,AARON,421,WHITT AVE,BURLINGTON,NC,27215,4,0.010164,0.000815,0.010107,0.010211,0.968703,WHITE,1.0,0.0,0.0
1,GENA,HOLT,AARONSON,107,TERRYWOOD CT,HAW RIVER,NC,27258,12,0.000425,0.005697,0.001222,0.003668,0.988988,WHITE,1.0,0.0,0.0
2,ANA-LINN,CHRISTINA,AASEN,1286,PYRTLE DR,GRAHAM,NC,27253,15,0.002188,0.001427,0.009616,0.612746,0.374024,HISPANIC,1.0,0.0,0.0
3,JACK,EDWARD,ABADIE,612,SIDEVIEW ST,GRAHAM,NC,27253,17,0.016160,0.000674,0.008744,0.114376,0.860045,WHITE,1.0,0.0,0.0
4,MYRA,LYNN,ABADIE,612,SIDEVIEW ST,GRAHAM,NC,27253,18,0.019254,0.000824,0.013051,0.182618,0.784253,WHITE,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,BRIAN,KEITH,POFF,4640,PAGETOWN RD #5,BURLINGTON,NC,27217,96133,0.000399,0.013262,0.001050,0.003354,0.981936,WHITE,1.0,0.0,0.0
9996,DAVID,JAMES,POGLITSCH,1618,GRACE LANDING DR,MEBANE,NC,27302,96136,0.001065,0.000373,0.003263,0.013453,0.981847,WHITE,1.0,0.0,0.0
9997,KEVIN,ANDREW,POGLITSCH,1803,MEADOW GREEN DR,GRAHAM,NC,27253,96138,0.051487,0.005903,0.030949,0.088708,0.822953,WHITE,0.0,0.0,1.0
9998,DAKARI,NATHAN SKYLER,POLINGER,1610,ELDER WAY,BURLINGTON,NC,27215,96176,0.002674,0.001786,0.332557,0.016247,0.646735,WHITE,1.0,0.0,0.0


In [ ]:
df.to_csv("py311_race_results.csv")